# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [6]:
# YOUR CODE HERE
# ============================================================
# NB5 — Solution V1 RAG Evaluation
# Shared Setup (3.3 preamble): load model, ChromaDB, test data,
# and define generate_baseline() / generate_naive_rag() helpers
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

import json
import pandas as pd
import os

DRIVE_DIR = "/content/drive/MyDrive/customer_support_llm_project/artifacts"

# ------------------------------------------------------------
# 1. Load outputs.json (baseline_output + naive_rag_output)
# ------------------------------------------------------------
with open(DRIVE_DIR + "/outputs.json", "r") as f:
    results = json.load(f)

print("Loaded outputs.json:", len(results), "records")
print("Schema:", list(results[0].keys()))

assert "baseline_output" in results[0], "baseline_output missing - check NB3 output"
assert "naive_rag_output" in results[0], "naive_rag_output missing - check NB4 output"

# ------------------------------------------------------------
# 2. Load df_test.csv (ground truth reference: text/intent/category)
# ------------------------------------------------------------
df_test = pd.read_csv(DRIVE_DIR + "/df_test.csv")
print("")
print("Loaded df_test:", df_test.shape)

assert len(df_test) == len(results), "Row count mismatch between df_test (" + str(len(df_test)) + ") and outputs.json (" + str(len(results)) + ")"

# ------------------------------------------------------------
# 3. Load Chroma index (must reuse same embedding model as NB4)
# ------------------------------------------------------------
!pip install -q langchain langchain-community langchain-chroma sentence-transformers chromadb bitsandbytes accelerate

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"  # must match NB4 exactly

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,
    model_kwargs={"device": "cpu"}
)

CHROMA_DIR = DRIVE_DIR + "/chroma_db"
vector_db = Chroma(persist_directory=CHROMA_DIR, embedding_function=embedding_model)

collection_count = vector_db._collection.count()
print("")
print("Loaded Chroma index from:", CHROMA_DIR)
print("Documents in collection:", collection_count)

assert collection_count > 0, "Chroma collection is empty - check NB4 persisted correctly"

test_retrieval = vector_db.similarity_search("refund", k=1)
print("Retrieval test - top result for 'refund':", test_retrieval[0].metadata["policy_name"])

# ------------------------------------------------------------
# 4. Load the base LLM (needed if any measurement below requires
#    fresh generation rather than re-scoring existing outputs.json)
# ------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # must match NB2, NB3, NB4 exactly

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("")
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

print("Model loaded. GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

# ------------------------------------------------------------
# 5. Shared generation helper (low-level, reused by both functions)
# ------------------------------------------------------------
def _generate(prompt_text, max_new_tokens=150):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ------------------------------------------------------------
# 6. generate_baseline() — no retrieval, matches NB3's JSON-intent
#    system prompt exactly (must stay identical for fair comparison)
# ------------------------------------------------------------
BASELINE_SYSTEM_PROMPT = (
    "You are a customer support intent classification assistant. "
    "Given a customer query, respond with ONLY a valid JSON object "
    "containing exactly two keys: \"intent\" and \"category\". "
    "Do not include any explanation, markdown formatting, or additional text."
)

def generate_baseline(query, max_new_tokens=120):
    messages = [
        {"role": "system", "content": BASELINE_SYSTEM_PROMPT},
        {"role": "user", "content": query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return _generate(prompt, max_new_tokens=max_new_tokens)

# ------------------------------------------------------------
# 7. generate_naive_rag() — retrieves top-k context, augments
#    prompt, matches NB4's RAG system prompt exactly
# ------------------------------------------------------------
RAG_SYSTEM_PROMPT_TEMPLATE = (
    "You are a helpful customer support assistant. "
    "Answer strictly using this SOP:\n\n{context}\n\n"
    "If the SOP does not contain the answer, say you don't have that information. "
    "Be concise."
)

def generate_naive_rag(query, k=1, max_new_tokens=150):
    retrieved_docs = vector_db.similarity_search(query, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    retrieved_policies = [doc.metadata["policy_name"] for doc in retrieved_docs]

    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT_TEMPLATE.format(context=context)},
        {"role": "user", "content": query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = _generate(prompt, max_new_tokens=max_new_tokens)

    return

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4
Loaded outputs.json: 391 records
Schema: ['test_query', 'ground_truth', 'baseline_output', 'naive_rag_output']

Loaded df_test: (391, 5)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━

/tmp/ipykernel_458/1635862310.py:47: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_458/1635862310.py:52: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. GPU memory allocated (GB): 1.15


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [7]:
# YOUR CODE HERE
# ============================================================
# Measurement 3.3.1 — Run Baseline and Naive RAG across full test set
# ============================================================



In [8]:
import inspect
print(inspect.getsource(generate_naive_rag))

def generate_naive_rag(query, k=1, max_new_tokens=150):
    retrieved_docs = vector_db.similarity_search(query, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    retrieved_policies = [doc.metadata["policy_name"] for doc in retrieved_docs]

    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT_TEMPLATE.format(context=context)},
        {"role": "user", "content": query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = _generate(prompt, max_new_tokens=max_new_tokens)

    return



In [9]:
def generate_naive_rag(query, k=1, max_new_tokens=150):
    retrieved_docs = vector_db.similarity_search(query, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    retrieved_policies = [doc.metadata["policy_name"] for doc in retrieved_docs]

    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT_TEMPLATE.format(context=context)},
        {"role": "user", "content": query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = _generate(prompt, max_new_tokens=max_new_tokens)

    return output, retrieved_policies


# Verify immediately before running anything else
test_check = generate_naive_rag(df_test["text"].iloc[0])
print("Type:", type(test_check))
print("Value:", test_check)
assert test_check is not None and isinstance(test_check, tuple) and len(test_check) == 2
print("CONFIRMED WORKING")

Type: <class 'tuple'>
Value: ("I'm sorry, but I am unable to assist with that. Please let me know if there's anything else I can help with.", ['product_return'])
CONFIRMED WORKING


In [10]:
import time
import json

RUN_FULL_SET = True
SAMPLE_SIZE = 50

if RUN_FULL_SET:
    eval_df = df_test.reset_index(drop=True)
else:
    eval_df = df_test.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print("Running evaluation on", len(eval_df), "rows")

evaluation_records = []
start_time = time.time()

for idx in range(len(eval_df)):
    query = eval_df["text"].iloc[idx]
    ground_truth_json = eval_df["json_target"].iloc[idx] if "json_target" in eval_df.columns else None
    true_intent = eval_df["intent"].iloc[idx]
    true_category = eval_df["category"].iloc[idx]

    baseline_out = generate_baseline(query)

    rag_result = generate_naive_rag(query)
    assert rag_result is not None, "generate_naive_rag returned None at row " + str(idx)
    assert isinstance(rag_result, tuple) and len(rag_result) == 2, "generate_naive_rag returned unexpected shape at row " + str(idx)
    naive_rag_out, retrieved_policies = rag_result

    evaluation_records.append({
        "query": query,
        "true_intent": true_intent,
        "true_category": true_category,
        "ground_truth_json": ground_truth_json,
        "baseline_output": baseline_out,
        "naive_rag_output": naive_rag_out,
        "retrieved_policies": retrieved_policies
    })

    if (idx + 1) % 25 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(eval_df) - (idx + 1)) / rate
        print("Processed", idx + 1, "/", len(eval_df), "- elapsed:", round(elapsed, 1), "s - est remaining:", round(remaining, 1), "s")

total_time = time.time() - start_time
print("")
print("Evaluation generation complete.")
print("Total rows:", len(evaluation_records))
print("Total time seconds:", round(total_time, 1))
print("Total time minutes:", round(total_time / 60, 1))
print("Avg time per row:", round(total_time / len(evaluation_records), 3))

print("")
print("--- Sample records ---")
for rec in evaluation_records[:2]:
    print("")
    print("Query:", rec["query"])
    print("True intent category:", rec["true_intent"], rec["true_category"])
    print("Baseline output:", rec["baseline_output"])
    print("Naive RAG output:", rec["naive_rag_output"])
    print("Retrieved policies:", rec["retrieved_policies"])

raw_eval_path = DRIVE_DIR + "/v1_raw_generations.json"
with open(raw_eval_path, "w") as f:
    json.dump(evaluation_records, f, indent=2)

print("")
print("Saved raw generation collection to:", raw_eval_path)

Running evaluation on 391 rows
Processed 25 / 391 - elapsed: 149.2 s - est remaining: 2183.9 s
Processed 50 / 391 - elapsed: 320.3 s - est remaining: 2184.2 s
Processed 75 / 391 - elapsed: 455.6 s - est remaining: 1919.8 s
Processed 100 / 391 - elapsed: 571.2 s - est remaining: 1662.3 s
Processed 125 / 391 - elapsed: 721.1 s - est remaining: 1534.6 s
Processed 150 / 391 - elapsed: 871.0 s - est remaining: 1399.3 s
Processed 175 / 391 - elapsed: 1013.3 s - est remaining: 1250.7 s
Processed 200 / 391 - elapsed: 1158.6 s - est remaining: 1106.4 s
Processed 225 / 391 - elapsed: 1289.8 s - est remaining: 951.6 s
Processed 250 / 391 - elapsed: 1437.2 s - est remaining: 810.6 s
Processed 275 / 391 - elapsed: 1580.6 s - est remaining: 666.7 s
Processed 300 / 391 - elapsed: 1730.0 s - est remaining: 524.8 s
Processed 325 / 391 - elapsed: 1885.3 s - est remaining: 382.9 s
Processed 350 / 391 - elapsed: 2036.5 s - est remaining: 238.6 s
Processed 375 / 391 - elapsed: 2170.9 s - est remaining: 92.

#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [11]:
# YOUR CODE HERE
# ============================================================
# Measurement 3.3.2 - Format Adherence Rate
# (Baseline / Naive RAG free-text outputs: well-formed and
# non-empty, NOT strict JSON - that check is reserved for NB7's
# fine-tuned router)
# ============================================================

def is_well_formed_response(text):
    if text is None:
        return False, "None value"

    stripped = text.strip()

    if len(stripped) == 0:
        return False, "empty string"

    if len(stripped) < 3:
        return False, "too short to be a real response"

    # Flag likely degenerate/repetition failures (a common small-model
    # failure mode: looping the same token or phrase until max_new_tokens)
    words = stripped.split()
    if len(words) >= 6:
        unique_ratio = len(set(words)) / len(words)
        if unique_ratio < 0.2:
            return False, "degenerate repetition detected"

    return True, "ok"


baseline_adherence_results = []
rag_adherence_results = []

for rec in evaluation_records:
    b_ok, b_reason = is_well_formed_response(rec["baseline_output"])
    r_ok, r_reason = is_well_formed_response(rec["naive_rag_output"])

    baseline_adherence_results.append({"query": rec["query"], "well_formed": b_ok, "reason": b_reason, "output": rec["baseline_output"]})
    rag_adherence_results.append({"query": rec["query"], "well_formed": r_ok, "reason": r_reason, "output": rec["naive_rag_output"]})

baseline_pass_count = sum(1 for r in baseline_adherence_results if r["well_formed"])
rag_pass_count = sum(1 for r in rag_adherence_results if r["well_formed"])

baseline_adherence_rate = 100 * baseline_pass_count / len(baseline_adherence_results)
rag_adherence_rate = 100 * rag_pass_count / len(rag_adherence_results)

print("--- Format Adherence Rate ---")
print("Baseline:", baseline_pass_count, "/", len(baseline_adherence_results), "=", round(baseline_adherence_rate, 2), "percent")
print("Naive RAG:", rag_pass_count, "/", len(rag_adherence_results), "=", round(rag_adherence_rate, 2), "percent")

print("")
print("--- Failure breakdown: Baseline ---")
baseline_failures = [r for r in baseline_adherence_results if not r["well_formed"]]
if baseline_failures:
    from collections import Counter
    reason_counts = Counter(r["reason"] for r in baseline_failures)
    for reason, count in reason_counts.items():
        print(" ", reason, ":", count)
    print("")
    print("Sample failures (up to 3):")
    for r in baseline_failures[:3]:
        print("  Query:", r["query"])
        print("  Output:", repr(r["output"]))
        print("  Reason:", r["reason"])
        print("")
else:
    print("No failures.")

print("--- Failure breakdown: Naive RAG ---")
rag_failures = [r for r in rag_adherence_results if not r["well_formed"]]
if rag_failures:
    from collections import Counter
    reason_counts = Counter(r["reason"] for r in rag_failures)
    for reason, count in reason_counts.items():
        print(" ", reason, ":", count)
    print("")
    print("Sample failures (up to 3):")
    for r in rag_failures[:3]:
        print("  Query:", r["query"])
        print("  Output:", repr(r["output"]))
        print("  Reason:", r["reason"])
        print("")
else:
    print("No failures.")

print("")
print("Format Adherence summary:")
print("  Baseline:", round(baseline_adherence_rate, 2), "percent")
print("  Naive RAG:", round(rag_adherence_rate, 2), "percent")

--- Format Adherence Rate ---
Baseline: 391 / 391 = 100.0 percent
Naive RAG: 391 / 391 = 100.0 percent

--- Failure breakdown: Baseline ---
No failures.
--- Failure breakdown: Naive RAG ---
No failures.

Format Adherence summary:
  Baseline: 100.0 percent
  Naive RAG: 100.0 percent


#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [12]:
# YOUR CODE HERE
import glob
import os

POLICY_DIR = "corporate_policies"
policy_files = sorted(glob.glob(os.path.join(POLICY_DIR, "*.md")))
policy_names = [os.path.splitext(os.path.basename(f))[0] for f in policy_files]

policy_full_text = {}
for fpath, pname in zip(policy_files, policy_names):
    with open(fpath, "r", encoding="utf-8") as f:
        policy_full_text[pname] = f.read()

all_intents = sorted(df_test["intent"].unique())
print("Unique intents in test set:", len(all_intents))
print(all_intents)

print("")
print("Available policy files:", policy_names)

# ------------------------------------------------------------
# Keyword-based auto-mapping attempt (intent -> policy_name)
# This is a STARTING POINT ONLY - review the printed output below
# and correct any wrong/missing mappings in INTENT_TO_POLICY_OVERRIDE
# ------------------------------------------------------------
def guess_policy_for_intent(intent, policy_names):
    intent_words = set(intent.replace("_", " ").split())
    best_match = None
    best_score = 0
    for pname in policy_names:
        policy_words = set(pname.replace("_", " ").split())
        overlap = len(intent_words & policy_words)
        if overlap > best_score:
            best_score = overlap
            best_match = pname
    return best_match, best_score

auto_mapping = {}
print("")
print("--- Auto-mapping attempt (intent -> policy, keyword overlap score) ---")
for intent in all_intents:
    guess, score = guess_policy_for_intent(intent, policy_names)
    auto_mapping[intent] = guess
    flag = "" if score > 0 else "  <-- NO KEYWORD OVERLAP, LIKELY WRONG, NEEDS MANUAL REVIEW"
    print(intent, "->", guess, "(score:", score, ")" + flag)

Unique intents in test set: 27
['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']

Available policy files: []

--- Auto-mapping attempt (intent -> policy, keyword overlap score) ---
cancel_order -> None (score: 0 )  <-- NO KEYWORD OVERLAP, LIKELY WRONG, NEEDS MANUAL REVIEW
change_order -> None (score: 0 )  <-- NO KEYWORD OVERLAP, LIKELY WRONG, NEEDS MANUAL REVIEW
change_shipping_address -> None (score: 0 )  <-- NO KEYWORD OVERLAP, LIKELY WRONG, NEEDS MANUAL REVIEW
check_cancellation_fee -> None (score: 0 )  <-- NO KEYWOR

#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [4]:
# YOUR CODE HERE
# ============================================================
# Measurement — Consistency Rate / Determinism check
# Same query, same generation settings, run 3x, confirm identical
# ============================================================
print("df_test defined:", "df_test" in dir())
print("model defined:", "model" in dir())
print("vector_db defined:", "vector_db" in dir())
print("generate_baseline defined:", "generate_baseline" in dir())
print("generate_naive_rag defined:", "generate_naive_rag" in dir())

df_test defined: False
model defined: False
vector_db defined: False
generate_baseline defined: False
generate_naive_rag defined: False


In [13]:
# ============================================================
# Measurement — Consistency Rate / Determinism check
# Same query, same generation settings, run 3x, confirm identical
# ============================================================

N_REPEATS = 3
N_TEST_QUERIES = 10  # sample a handful of queries, not just one, for a more robust check

sample_queries = df_test["text"].sample(N_TEST_QUERIES, random_state=42).tolist()

consistency_results = []

for query in sample_queries:
    baseline_runs = [generate_baseline(query) for _ in range(N_REPEATS)]
    rag_runs = [generate_naive_rag(query)[0] for _ in range(N_REPEATS)]  # [0] = output text, drop retrieved_policies

    baseline_identical = len(set(baseline_runs)) == 1
    rag_identical = len(set(rag_runs)) == 1

    consistency_results.append({
        "query": query,
        "baseline_runs": baseline_runs,
        "baseline_identical": baseline_identical,
        "rag_runs": rag_runs,
        "rag_identical": rag_identical
    })

print("--- Determinism check across", N_TEST_QUERIES, "queries x", N_REPEATS, "repeats ---")
print("")

baseline_consistent_count = sum(1 for r in consistency_results if r["baseline_identical"])
rag_consistent_count = sum(1 for r in consistency_results if r["rag_identical"])

for i, r in enumerate(consistency_results):
    print("Query", i + 1, ":", r["query"])
    print("  Baseline identical across", N_REPEATS, "runs:", r["baseline_identical"])
    if not r["baseline_identical"]:
        for j, run in enumerate(r["baseline_runs"]):
            print("    Run", j + 1, ":", repr(run))
    print("  Naive RAG identical across", N_REPEATS, "runs:", r["rag_identical"])
    if not r["rag_identical"]:
        for j, run in enumerate(r["rag_runs"]):
            print("    Run", j + 1, ":", repr(run))
    print("")

baseline_consistency_rate = 100 * baseline_consistent_count / N_TEST_QUERIES
rag_consistency_rate = 100 * rag_consistent_count / N_TEST_QUERIES

print("--- Consistency Rate summary ---")
print("Baseline:", baseline_consistent_count, "/", N_TEST_QUERIES, "=", round(baseline_consistency_rate, 1), "percent")
print("Naive RAG:", rag_consistent_count, "/", N_TEST_QUERIES, "=", round(rag_consistency_rate, 1), "percent")

if baseline_consistency_rate < 100 or rag_consistency_rate < 100:
    print("")
    print("WARNING: Expected 100 percent determinism with do_sample=False, temperature=None.")
    print("Non-determinism at this setting usually comes from one of:")
    print("  - GPU floating-point non-associativity (rare but possible on some CUDA kernels/batch sizes)")
    print("  - A stray default reintroducing sampling somewhere in the generation call")
    print("  - Retrieval randomness (unlikely with Chroma's deterministic similarity search, but worth checking k results order)")
else:
    print("")
    print("Fully deterministic - all outputs identical across repeated runs, as expected for greedy decoding.")

--- Determinism check across 10 queries x 3 repeats ---

Query 1 : I cannot afford order {{Order Number}}
  Baseline identical across 3 runs: True
  Naive RAG identical across 3 runs: True

Query 2 : I need assistance to enter another shipping address
  Baseline identical across 3 runs: True
  Naive RAG identical across 3 runs: True

Query 3 : need assistance changing the details on my profile
  Baseline identical across 3 runs: True
  Naive RAG identical across 3 runs: True

Query 4 : I do not know what I have to do to speak with an operator
  Baseline identical across 3 runs: True
  Naive RAG identical across 3 runs: True

Query 5 : I have problems entering a different shipping address
  Baseline identical across 3 runs: True
  Naive RAG identical across 3 runs: True

Query 6 : help me cancel my subscriptiopn to ur newsletter
  Baseline identical across 3 runs: True
  Naive RAG identical across 3 runs: True

Query 7 : ohw do I check how soon can I expect the item?
  Baseline identica

In [16]:
print("df_test defined:", "df_test" in dir())
print("evaluation_records defined:", "evaluation_records" in dir())
if "evaluation_records" in dir():
    print("evaluation_records length:", len(evaluation_records))
    print("Local corporate_policies exists:", __import__("os").path.isdir("corporate_policies"))
!find /content/drive/MyDrive -iname "*.md" 2>/dev/null

df_test defined: True
evaluation_records defined: True
evaluation_records length: 391
Local corporate_policies exists: False
/content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies/account_recovery.md
/content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies/subscription_cancellation.md
/content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies/payment_methods.md
/content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies/data_privacy.md
/content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies/escalation_matrix.md
/content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies/billing_disputes.md
/content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies/order_tracking.md
/content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies/password_reset.md
/content/drive/MyDrive/customer_support_llm_project/artifacts/corp

In [15]:
!find /content/drive/MyDrive -iname "*.md" 2>/dev/null

#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [17]:
# YOUR CODE HERE
import glob
import os

POLICY_DIR = "/content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies"

policy_files = sorted(glob.glob(os.path.join(POLICY_DIR, "*.md")))
print("Found", len(policy_files), "policy files at:", POLICY_DIR)

assert len(policy_files) == 13, "Expected 13 policy files, found " + str(len(policy_files))

policy_full_text = {}
for fpath in policy_files:
    pname = os.path.splitext(os.path.basename(fpath))[0]
    with open(fpath, "r", encoding="utf-8") as f:
        policy_full_text[pname] = f.read()

print("Loaded policy text for:", list(policy_full_text.keys()))

Found 13 policy files at: /content/drive/MyDrive/customer_support_llm_project/artifacts/corporate_policies
Loaded policy text for: ['account_recovery', 'billing_disputes', 'data_privacy', 'escalation_matrix', 'order_tracking', 'password_reset', 'payment_methods', 'product_return', 'refund_policy', 'shipping_delays', 'subscription_cancellation', 'technical_troubleshooting', 'working_hours']


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [18]:
# YOUR CODE HERE
# ============================================================
# Aggregate comparison: Baseline vs Naive RAG
# Functional Correctness (Format Adherence), Consistency, Hallucination Frequency
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Pull in whatever's already been computed in this session from
# the earlier measurements. Check each exists before building
# the table - don't silently plug in zeros or guesses.
# ------------------------------------------------------------
required_vars = [
    "baseline_adherence_rate", "rag_adherence_rate",
    "baseline_consistency_rate", "rag_consistency_rate",
    "baseline_halluc_rate", "rag_halluc_rate"
]

missing = [v for v in required_vars if v not in dir()]

print("--- Variable availability check ---")
for v in required_vars:
    print(v, ":", "OK" if v in dir() else "MISSING - re-run that measurement's cell first")

if missing:
    print("")
    print("Cannot build the summary table yet - missing:", missing)
    print("Re-run the corresponding measurement cell(s) above before proceeding.")
else:
    def pct_change(baseline_val, rag_val):
        if baseline_val == 0:
            return float("inf") if rag_val > 0 else 0.0
        return (rag_val - baseline_val) / baseline_val * 100

    summary_rows = [
        {
            "Metric": "Functional Correctness (Format Adherence %)",
            "Baseline": round(baseline_adherence_rate, 2),
            "Naive RAG": round(rag_adherence_rate, 2),
            "Change (%)": round(pct_change(baseline_adherence_rate, rag_adherence_rate), 2)
        },
        {
            "Metric": "Consistency Rate (%)",
            "Baseline": round(baseline_consistency_rate, 2),
            "Naive RAG": round(rag_consistency_rate, 2),
            "Change (%)": round(pct_change(baseline_consistency_rate, rag_consistency_rate), 2)
        },
        {
            "Metric": "Hallucination Frequency (%)",
            "Baseline": round(baseline_halluc_rate, 2),
            "Naive RAG": round(rag_halluc_rate, 2),
            # For hallucination, a DECREASE is improvement - flip sign convention
            # so "positive change" consistently means "better" across all three rows
            "Change (%, negative = improvement)": round(pct_change(baseline_halluc_rate, rag_halluc_rate), 2)
        },
    ]

    summary_df = pd.DataFrame(summary_rows)
    print("")
    print("=== Baseline vs Naive RAG: Aggregate Comparison ===")
    print(summary_df.to_string(index=False))

    print("")
    print("--- Interpretation notes ---")
    if rag_adherence_rate > baseline_adherence_rate:
        print("- Retrieval IMPROVED format adherence: grounding in SOP text produced more well-formed responses.")
    elif rag_adherence_rate < baseline_adherence_rate:
        print("- Retrieval REDUCED format adherence - possibly longer/more complex prompts introduced more failure modes.")
    else:
        print("- Format adherence UNCHANGED by retrieval.")

    if rag_consistency_rate >= baseline_consistency_rate:
        print("- Consistency held steady or improved with retrieval (expected, since both use greedy decoding).")
    else:
        print("- Consistency DROPPED with retrieval - worth investigating (e.g. retrieval order changing between runs).")

    if rag_halluc_rate < baseline_halluc_rate:
        print("- Retrieval REDUCED hallucination frequency: grounding in real SOP text cut down fabricated claims.")
    elif rag_halluc_rate > baseline_halluc_rate:
        print("- Retrieval INCREASED hallucination frequency - possible cause: irrelevant/wrong-policy retrieval")
        print("  introducing distracting context, or the model ignoring the provided SOP and inventing anyway.")
    else:
        print("- Hallucination frequency UNCHANGED by retrieval.")

    # ------------------------------------------------------------
    # Save as part of the notebook's audit trail (this feeds into
    # v1_metrics.csv alongside the per-row metrics from earlier steps)
    # ------------------------------------------------------------
    aggregate_summary_path = DRIVE_DIR + "/v1_aggregate_summary.csv"
    summary_df.to_csv(aggregate_summary_path, index=False)
    print("")
    print("Saved aggregate summary to:", aggregate_summary_path)

--- Variable availability check ---
baseline_adherence_rate : OK
rag_adherence_rate : OK
baseline_consistency_rate : OK
rag_consistency_rate : OK
baseline_halluc_rate : MISSING - re-run that measurement's cell first
rag_halluc_rate : MISSING - re-run that measurement's cell first

Cannot build the summary table yet - missing: ['baseline_adherence_rate', 'rag_adherence_rate', 'baseline_consistency_rate', 'rag_consistency_rate', 'baseline_halluc_rate', 'rag_halluc_rate']
Re-run the corresponding measurement cell(s) above before proceeding.


In [19]:
import re

def extract_specific_claims(text):
    numbers = re.findall(r'\b\d+(?:-\d+)?\b', text)
    return numbers

def check_hallucination(output_text, sop_context):
    claims = extract_specific_claims(output_text)
    unsupported = [c for c in claims if c not in sop_context]
    return unsupported

hallucination_results = []

for rec in evaluation_records:
    query = rec["query"]
    retrieved_docs = vector_db.similarity_search(query, k=1)
    sop_context = retrieved_docs[0].page_content if retrieved_docs else ""

    baseline_unsupported = check_hallucination(rec["baseline_output"], sop_context)
    rag_unsupported = check_hallucination(rec["naive_rag_output"], sop_context)

    hallucination_results.append({
        "query": query,
        "retrieved_policy": retrieved_docs[0].metadata["policy_name"] if retrieved_docs else None,
        "baseline_output": rec["baseline_output"],
        "baseline_unsupported_numbers": baseline_unsupported,
        "baseline_hallucinated": len(baseline_unsupported) > 0,
        "naive_rag_output": rec["naive_rag_output"],
        "rag_unsupported_numbers": rag_unsupported,
        "rag_hallucinated": len(rag_unsupported) > 0,
    })

baseline_halluc_count = sum(1 for r in hallucination_results if r["baseline_hallucinated"])
rag_halluc_count = sum(1 for r in hallucination_results if r["rag_hallucinated"])

baseline_halluc_rate = 100 * baseline_halluc_count / len(hallucination_results)
rag_halluc_rate = 100 * rag_halluc_count / len(hallucination_results)

print("Baseline hallucination rate:", round(baseline_halluc_rate, 2), "percent")
print("Naive RAG hallucination rate:", round(rag_halluc_rate, 2), "percent")

Baseline hallucination rate: 0.0 percent
Naive RAG hallucination rate: 9.97 percent


In [20]:
rag_flagged = [r for r in hallucination_results if r["rag_hallucinated"]]
print("Total flagged:", len(rag_flagged))
print("")
for r in rag_flagged[:8]:
    print("Query:", r["query"])
    print("Retrieved policy:", r["retrieved_policy"])
    print("RAG output:", r["naive_rag_output"])
    print("Flagged numbers:", r["rag_unsupported_numbers"])
    print("")

Total flagged: 39

Query: how can I get information about opening a standard account?
Retrieved policy: data_privacy
RAG output: To open a standard account, please follow these steps:

1. Verify your identity by providing necessary documents (e.g., ID, proof of address).
2. Fill out the application form online or at a physical location.
3. Submit the completed application along with required documentation.
4. Wait for approval; typically, this process takes approximately 5 business days from submission.

Please ensure all required documents are accurate and complete when submitting your application. If you encounter any issues during the process, feel free to reach out to our support team for assistance.
Flagged numbers: ['5']

Query: show me in which situations can I ask for reimbursements
Retrieved policy: product_return
RAG output: The refund policy allows customers to request reimbursement under certain circumstances. These include:
1. **Defective Items**: If the product received i

In [22]:
# ============================================================
# Corrected hallucination detection:
#  1. Strip markdown list-numbering before extracting claims
#  2. Exclude numbers that appear verbatim in the user's own query
#     (echoing the user is not fabrication)
#  3. Show retrieved SOP text alongside flagged claims for manual
#     confirmation, rather than trusting the automated flag alone
# ============================================================

import re

def extract_specific_claims(text):
    # Remove markdown list markers ("1. ", "2. " etc at line starts)
    # so list-item indices are not mistaken for factual claims
    cleaned = re.sub(r'^\s*\d+\.\s+', '', text, flags=re.MULTILINE)
    numbers = re.findall(r'\b\d+(?:-\d+)?\b', cleaned)
    return numbers


def check_hallucination_v2(output_text, sop_context, query_text):
    claims = extract_specific_claims(output_text)

    # Exclude anything already present in the user's own query
    # (echoing back "bill #00108" is not a hallucination)
    query_numbers = set(re.findall(r'\b\d+(?:-\d+)?\b', query_text))

    unsupported = [
        c for c in claims
        if c not in sop_context and c not in query_numbers
    ]
    return unsupported


hallucination_results_v2 = []

for rec in evaluation_records:
    query = rec["query"]
    retrieved_docs = vector_db.similarity_search(query, k=1)
    sop_context = retrieved_docs[0].page_content if retrieved_docs else ""
    retrieved_policy = retrieved_docs[0].metadata["policy_name"] if retrieved_docs else None

    baseline_unsupported = check_hallucination_v2(rec["baseline_output"], sop_context, query)
    rag_unsupported = check_hallucination_v2(rec["naive_rag_output"], sop_context, query)

    hallucination_results_v2.append({
        "query": query,
        "retrieved_policy": retrieved_policy,
        "retrieved_sop_text": sop_context,
        "baseline_output": rec["baseline_output"],
        "baseline_unsupported_numbers": baseline_unsupported,
        "baseline_hallucinated": len(baseline_unsupported) > 0,
        "naive_rag_output": rec["naive_rag_output"],
        "rag_unsupported_numbers": rag_unsupported,
        "rag_hallucinated": len(rag_unsupported) > 0,
    })

baseline_halluc_count = sum(1 for r in hallucination_results_v2 if r["baseline_hallucinated"])
rag_halluc_count = sum(1 for r in hallucination_results_v2 if r["rag_hallucinated"])

baseline_halluc_rate = 100 * baseline_halluc_count / len(hallucination_results_v2)
rag_halluc_rate = 100 * rag_halluc_count / len(hallucination_results_v2)

print("--- Corrected Hallucination Frequency ---")
print("Baseline:", baseline_halluc_count, "/", len(hallucination_results_v2), "=", round(baseline_halluc_rate, 2), "percent")
print("Naive RAG:", rag_halluc_count, "/", len(hallucination_results_v2), "=", round(rag_halluc_rate, 2), "percent")
print("")
print("(Previous uncorrected rates: Baseline 0.0 percent, Naive RAG 9.97 percent)")

# ------------------------------------------------------------
# Show ALL flagged RAG cases with SOP context side-by-side,
# for manual confirmation before finalizing the number
# ------------------------------------------------------------
rag_flagged_v2 = [r for r in hallucination_results_v2 if r["rag_hallucinated"]]

print("")
print("Total flagged after correction:", len(rag_flagged_v2))
print("")
print("--- Full detail: retrieved SOP text vs flagged claim ---")
for i, r in enumerate(rag_flagged_v2):
    print(str(i + 1) + ". Query:", r["query"])
    print("   Retrieved policy:", r["retrieved_policy"])
    print("   Flagged numbers:", r["rag_unsupported_numbers"])
    print("   RAG output:", r["naive_rag_output"])
    print("   --- Retrieved SOP text (for manual check) ---")
    print("  ", r["retrieved_sop_text"][:500])
    print("")

--- Corrected Hallucination Frequency ---
Baseline: 0 / 391 = 0.0 percent
Naive RAG: 18 / 391 = 4.6 percent

(Previous uncorrected rates: Baseline 0.0 percent, Naive RAG 9.97 percent)

Total flagged after correction: 18

--- Full detail: retrieved SOP text vs flagged claim ---
1. Query: how can I get information about opening a standard account?
   Retrieved policy: data_privacy
   Flagged numbers: ['5']
   RAG output: To open a standard account, please follow these steps:

1. Verify your identity by providing necessary documents (e.g., ID, proof of address).
2. Fill out the application form online or at a physical location.
3. Submit the completed application along with required documentation.
4. Wait for approval; typically, this process takes approximately 5 business days from submission.

Please ensure all required documents are accurate and complete when submitting your application. If you encounter any issues during the process, feel free to reach out to our support team for assi

---
## Save Artifacts

In [25]:
print("df_test:", "df_test" in dir())
print("evaluation_records:", "evaluation_records" in dir())
print("model:", "model" in dir())
print("vector_db:", "vector_db" in dir())
print("generate_baseline:", "generate_baseline" in dir())
print("generate_naive_rag:", "generate_naive_rag" in dir())
print("is_well_formed_response:", "is_well_formed_response" in dir())
print("hallucination_results_v2:", "hallucination_results_v2" in dir())
print("baseline_adherence_rate:", "baseline_adherence_rate" in dir())
print("baseline_consistency_rate:", "baseline_consistency_rate" in dir())

df_test: True
evaluation_records: True
model: True
vector_db: True
generate_baseline: True
generate_naive_rag: True
is_well_formed_response: True
hallucination_results_v2: True
baseline_adherence_rate: True
baseline_consistency_rate: True


In [26]:
print("rag_adherence_rate:", "rag_adherence_rate" in dir())
print("rag_consistency_rate:", "rag_consistency_rate" in dir())
print("baseline_halluc_rate:", "baseline_halluc_rate" in dir())
print("rag_halluc_rate:", "rag_halluc_rate" in dir())

rag_adherence_rate: True
rag_consistency_rate: True
baseline_halluc_rate: True
rag_halluc_rate: True


In [28]:
import pandas as pd
import json
import os

DRIVE_DIR = "/content/drive/MyDrive/customer_support_llm_project/artifacts"
os.makedirs(DRIVE_DIR, exist_ok=True)

current_names = dir()

required_vars = ["baseline_adherence_rate", "rag_adherence_rate", "baseline_consistency_rate", "rag_consistency_rate", "baseline_halluc_rate", "rag_halluc_rate"]
missing = [v for v in required_vars if v not in current_names]
assert len(missing) == 0, "Missing required variables: " + str(missing)

print("All required variables confirmed present.")

per_row_records = []

for i in range(len(evaluation_records)):
    rec = evaluation_records[i]
    b_wellformed, b_reason = is_well_formed_response(rec["baseline_output"])
    r_wellformed, r_reason = is_well_formed_response(rec["naive_rag_output"])

    halluc_row = hallucination_results_v2[i]

    row = {}
    row["query"] = rec["query"]
    row["true_intent"] = rec["true_intent"]
    row["true_category"] = rec["true_category"]
    row["baseline_output"] = rec["baseline_output"]
    row["baseline_well_formed"] = b_wellformed
    row["baseline_hallucinated"] = halluc_row["baseline_hallucinated"]
    row["naive_rag_output"] = rec["naive_rag_output"]
    row["naive_rag_well_formed"] = r_wellformed
    row["naive_rag_hallucinated"] = halluc_row["rag_hallucinated"]
    row["retrieved_policy"] = halluc_row["retrieved_policy"]

    per_row_records.append(row)

v1_metrics_df = pd.DataFrame(per_row_records)

v1_metrics_path = DRIVE_DIR + "/v1_metrics.csv"
v1_metrics_df.to_csv(v1_metrics_path, index=False)

print("Saved v1_metrics.csv to:", v1_metrics_path)
print("Shape:", v1_metrics_df.shape)
print("")
print(v1_metrics_df.head(3))


def pct_change(baseline_val, rag_val):
    if baseline_val == 0:
        if rag_val > 0:
            return float("inf")
        else:
            return 0.0
    return (rag_val - baseline_val) / baseline_val * 100.0


row1 = {}
row1["Metric"] = "Format Adherence Percent"
row1["Baseline"] = round(baseline_adherence_rate, 2)
row1["Naive_RAG"] = round(rag_adherence_rate, 2)
row1["Change_Percent"] = round(pct_change(baseline_adherence_rate, rag_adherence_rate), 2)

row2 = {}
row2["Metric"] = "Consistency Rate Percent"
row2["Baseline"] = round(baseline_consistency_rate, 2)
row2["Naive_RAG"] = round(rag_consistency_rate, 2)
row2["Change_Percent"] = round(pct_change(baseline_consistency_rate, rag_consistency_rate), 2)

row3 = {}
row3["Metric"] = "Hallucination Frequency Percent"
row3["Baseline"] = round(baseline_halluc_rate, 2)
row3["Naive_RAG"] = round(rag_halluc_rate, 2)
row3["Change_Percent"] = round(pct_change(baseline_halluc_rate, rag_halluc_rate), 2)

aggregate_rows = [row1, row2, row3]
aggregate_summary_df = pd.DataFrame(aggregate_rows)

aggregate_path = DRIVE_DIR + "/v1_aggregate_summary.csv"
aggregate_summary_df.to_csv(aggregate_path, index=False)

print("")
print("Saved v1_aggregate_summary.csv to:", aggregate_path)
print(aggregate_summary_df.to_string(index=False))
print("")
print("Note: for Hallucination Frequency, a NEGATIVE Change Percent means improvement (fewer hallucinations).")

halluc_detail_path = DRIVE_DIR + "/v1_hallucination_detail.json"
with open(halluc_detail_path, "w") as f:
    json.dump(hallucination_results_v2, f, indent=2)

print("")
print("Saved hallucination detail to:", halluc_detail_path)

reloaded_v1 = pd.read_csv(v1_metrics_path)
assert len(reloaded_v1) == len(evaluation_records), "v1_metrics.csv reload size mismatch!"
assert "baseline_hallucinated" in reloaded_v1.columns, "Missing column after reload!"

print("")
print("Reload verification PASSED.")
print("")
print("All NB5 artifacts saved:")
print("  -", v1_metrics_path, "(required output)")
print("  -", aggregate_path, "(supporting summary)")
print("  -", halluc_detail_path, "(supporting detail and appendix)")

All required variables confirmed present.
Saved v1_metrics.csv to: /content/drive/MyDrive/customer_support_llm_project/artifacts/v1_metrics.csv
Shape: (391, 10)

                                               query              true_intent  \
0               need help to shop several of ur item              place_order   
1  i need assistance trying to update the shippin...  change_shipping_address   
2  I do not know how to check the cancellation  p...   check_cancellation_fee   

  true_category                                    baseline_output  \
0         ORDER  ```json\n{\n  "intent": "customer_support",\n ...   
1      SHIPPING  {\n  "intent": "customer service",\n  "categor...   
2        CANCEL  {\n  "intent": "information_request",\n  "cate...   

   baseline_well_formed  baseline_hallucinated  \
0                  True                  False   
1                  True                  False   
2                  True                  False   

                               

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**